# Adım 2 — Isınma eğitimi: küçük bir veriyle ilk güvenlik görevlisini eğitelim

**Amaç:** Büyük deneye girmeden önce "boru hattı çalışıyor mu?" diye görmek. Küçük bir veri setiyle BERT'i eğitip ilk kez bir karne çıkaracağız.

**Önemli uyarı:** Bu veri setinin test kısmı çok küçük (yüz küsur cümle). Çıkan sonuç **güzel görünse bile güvenilir değildir.** Bildiriye bu sayıyı koymayacağız. Bu sadece "kod çalışıyor" kanıtı.

**Önce:** `Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU` seçili olsun. Hücreleri sırayla çalıştırın. Hata çıkarsa hatayı kopyalayıp Claude'a yapıştırın.

In [ ]:
!pip -q install transformers datasets scikit-learn

## 1) Gerekli parçaları çağıralım

In [ ]:
import random
import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix)

# Her çalıştırmada aynı sonucu almak için rastgeleliği sabitliyoruz
TOHUM = 42
random.seed(TOHUM); np.random.seed(TOHUM); torch.manual_seed(TOHUM)

cihaz = "cuda" if torch.cuda.is_available() else "cpu"   # GPU varsa onu kullan
print("Kullanılan cihaz:", cihaz)   # 'cuda' görmelisiniz. 'cpu' ise GPU seçmeyi unutmuşsunuz demektir.

## 2) Veriyi indirelim ve bakalım

In [ ]:
ds = load_dataset("deepset/prompt-injections")
print(ds)                      # hangi bölümler ve kaç satır var
print(ds["train"][0])          # ilk satır neye benziyor?

# Sütun adlarını otomatik bulalım (veri setine göre farklı olabiliyor)
sutunlar = ds["train"].column_names
METIN_SUTUNU = next(s for s in sutunlar if s.lower() in ("text", "prompt", "sentence"))
ETIKET_SUTUNU = next(s for s in sutunlar if s.lower() in ("label", "labels", "type"))
print("Metin sütunu:", METIN_SUTUNU, "| Etiket sütunu:", ETIKET_SUTUNU)

## 3) Modeli ve "kelime parçalayıcıyı" (tokenizer) yükleyelim
Model metni doğrudan okuyamaz, sayılara çevrilmesi gerekir. Bu işi **tokenizer** yapar.
`MODEL_ADI` kısmını sonra `"roberta-base"` yapıp aynı kodu RoBERTa için de çalıştıracağız.

In [ ]:
MODEL_ADI = "bert-base-uncased"     # sonra: "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ADI)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_ADI, num_labels=2)  # 2 sınıf: zararsız / zararlı
model.to(cihaz)

## 4) Metinleri sayıya çevirip eğitime hazırlayalım

In [ ]:
MAKS_UZUNLUK = 128   # çok uzun metinler bu kadar parçada kesilir

def hazirla(bolum):
    metinler = list(bolum[METIN_SUTUNU])
    etiketler = [int(e) for e in bolum[ETIKET_SUTUNU]]       # etiketler 0 veya 1 olmalı
    kodlanmis = tokenizer(metinler, padding="max_length", truncation=True,
                          max_length=MAKS_UZUNLUK, return_tensors="pt")
    return TensorDataset(kodlanmis["input_ids"], kodlanmis["attention_mask"], torch.tensor(etiketler)), metinler, etiketler

egitim_verisi, _, egitim_etiketleri = hazirla(ds["train"])
test_verisi, test_metinleri, test_etiketleri = hazirla(ds["test"])
print("Eğitim:", len(egitim_verisi), "örnek | Test:", len(test_verisi), "örnek")
print("Eğitimde zararlı oranı:", sum(egitim_etiketleri) / len(egitim_etiketleri))

## 5) Eğitim
Model her turda (epoch) tüm eğitim örneklerini görür, tahmin yapar, ne kadar yanıldığına bakar (**loss**) ve kendini biraz düzeltir.
Loss'un tur tur **düşmesi** iyi işarettir.

In [ ]:
TUR = 3              # veriyi kaç kez baştan sona gezeceği
PARTI = 16           # aynı anda kaç örnek işlenecek
OGRENME_HIZI = 2e-5  # her adımda ne kadar düzeltme yapılacağı (küçük tutulur)

yukleyici = DataLoader(egitim_verisi, batch_size=PARTI, shuffle=True)
optimizer = torch.optim.AdamW(model.parameters(), lr=OGRENME_HIZI)

for tur in range(TUR):
    model.train()
    toplam_loss = 0
    for ids, maske, etiket in yukleyici:
        ids, maske, etiket = ids.to(cihaz), maske.to(cihaz), etiket.to(cihaz)
        cikti = model(input_ids=ids, attention_mask=maske, labels=etiket)  # tahmin + hata hesabı
        cikti.loss.backward()      # hatanın nereden geldiğini hesapla
        optimizer.step()           # modeli düzelt
        optimizer.zero_grad()      # bir sonraki adım için sıfırla
        toplam_loss += cikti.loss.item()
    print(f"Tur {tur+1}/{TUR} | ortalama loss: {toplam_loss/len(yukleyici):.4f}")

## 6) Test: karne zamanı

In [ ]:
model.eval()
tahminler, olasiliklar = [], []
with torch.no_grad():                       # test sırasında öğrenme yok
    for ids, maske, etiket in DataLoader(test_verisi, batch_size=32):
        mantik = model(input_ids=ids.to(cihaz), attention_mask=maske.to(cihaz)).logits
        p = torch.softmax(mantik, dim=1)[:, 1]          # "zararlı olma ihtimali"
        olasiliklar += p.cpu().tolist()
        tahminler += mantik.argmax(dim=1).cpu().tolist()

gercek = test_etiketleri
print("Doğruluk (accuracy):", round(accuracy_score(gercek, tahminler), 4))
print("Kesinlik (precision):", round(precision_score(gercek, tahminler, zero_division=0), 4))
print("Duyarlılık (recall):", round(recall_score(gercek, tahminler, zero_division=0), 4))
print("F1:", round(f1_score(gercek, tahminler, zero_division=0), 4))
print("ROC-AUC:", round(roc_auc_score(gercek, olasiliklar), 4))
tn, fp, fn, tp = confusion_matrix(gercek, tahminler).ravel()
print(f"Karışıklık tablosu → doğru yakalanan zararlı: {tp} | kaçırılan zararlı: {fn} | yanlışlıkla engellenen masum: {fp} | doğru geçen masum: {tn}")

## 7) Model nerede yanıldı? (Hata analizi)

In [ ]:
yanlislar = [i for i, (g, t) in enumerate(zip(gercek, tahminler)) if g != t]
print("Toplam yanlış:", len(yanlislar))
for i in yanlislar[:8]:
    tur = "KAÇIRILAN ZARARLI" if gercek[i] == 1 else "YANLIŞ ENGELLENEN MASUM"
    print(f"- [{tur}] {test_metinleri[i][:200]!r}")

## 8) Çıktıyı Claude'a yapıştırın ✅
6. ve 7. hücrenin çıktısını (karne + hata listesi) kopyalayıp Claude'a gönderin.
Ardından `MODEL_ADI = "roberta-base"` yapıp 3'ten itibaren hücreleri tekrar çalıştırın ve onun karnesini de gönderin.